![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/banner_1.png)

# Proyecto 2 - Clasificación de género de películas

El propósito de este proyecto es que puedan poner en práctica, en sus respectivos grupos de trabajo, sus conocimientos sobre técnicas de preprocesamiento, modelos predictivos de NLP, y la disponibilización de modelos. Para su desarrollo tengan en cuenta las instrucciones dadas en la "Guía del proyecto 2: Clasificación de género de películas"

Para hacer la entrega, deberán adjuntar el informe autocontenido en PDF a la actividad de entrega del proyecto que encontrarán en la semana 8, y subir el archivo de predicciones a la [competencia de Kaggle](https://www.kaggle.com/t/29c44fce98c747f2a1dfdaf29d4c4965).

En este proyecto se usará un conjunto de datos de géneros de películas. Cada observación contiene el título de una película, su año de lanzamiento, la sinopsis o plot de la película (resumen de la trama) y los géneros a los que pertenece (una película puede pertenercer a más de un género). Por ejemplo:
- Título: 'How to Be a Serial Killer'
- Plot: 'A serial killer decides to teach the secrets of his satisfying career to a video store clerk.'
- Generos: 'Comedy', 'Crime', 'Horror'

La idea es que usen estos datos para predecir la probabilidad de que una película pertenezca, dada la sinopsis, a cada uno de los géneros.

![image info](https://raw.githubusercontent.com/albahnsen/MIAD_ML_and_NLP/main/images/moviegenre.png)

### Librerías

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [10]:
# Importación librerías
import pandas as pd
import os
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import r2_score, roc_auc_score
from sklearn.model_selection import train_test_split

# Adicionales
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from scipy.sparse import hstack, csr_matrix
#from sentence_transformers import SentenceTransformer
from google.colab import files

### Datos para la predicción de género en películas

In [3]:
# Carga de datos de archivo .csv
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)
dataTesting = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip', encoding='UTF-8', index_col=0)

In [4]:
# Visualización datos de entrenamiento
dataTraining.head()

,year,title,plot,genres,rating
3107,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0
900,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6
6724,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2
4704,1954,Executive Suite,"in a friday afternoon in new york , the presi...",['Drama'],7.4
2582,1990,Narrow Margin,"in los angeles , the editor of a publishing h...","['Action', 'Crime', 'Thriller']",6.6


In [5]:
# Visualización datos de test
dataTesting.head()

,year,title,plot
1,1999,Message in a Bottle,"who meets by fate , shall be sealed by fate ...."
4,1978,Midnight Express,"the true story of billy hayes , an american c..."
5,1996,Primal Fear,martin vail left the chicago da ' s office to ...
6,1950,Crisis,husband and wife americans dr . eugene and mr...
7,1959,The Tingler,the coroner and scientist dr . warren chapin ...


## Regresión Logística + TF-IDF

In [6]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [7]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

print(y.shape)

(7895, 24)


In [8]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(dataTraining['text'],
                                                            y,
                                                            test_size=0.2,
                                                            random_state=42)

In [19]:
# Definición y entrenamiento

## TF-IDF Word
vect = TfidfVectorizer(lowercase=True,
                       stop_words='english',
                       max_features=70000,
                       ngram_range=(1,3),
                       min_df=2,
                       max_df=0.90,
                       sublinear_tf=True,
                       dtype=float32)

X_train_word = vect.fit_transform(X_train_text)

X_test_word = vect.transform(X_test_text)

print(X_train_word.shape)
print(X_test_word.shape)

## Numéricas
X_num_train = dataTraining.loc[X_train_text.index, ['year']]
X_num_test = dataTraining.loc[X_test_text.index, ['year']]

X_num_train = X_num_train.fillna(0)
X_num_test = X_num_test.fillna(0)

scaler = StandardScaler()

X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled = scaler.transform(X_num_test)

X_num_train_sparse = csr_matrix(X_num_train_scaled)
X_num_test_sparse = csr_matrix(X_num_test_scaled)

## Combinación
X_train_final = hstack([X_train_word, X_num_train_sparse])

X_test_final = hstack([X_test_word, X_num_test_sparse])

print(X_train_final.shape)
print(X_test_final.shape)

## Modelo
clf = OneVsRestClassifier(LogisticRegression(C=3,
                                             solver='liblinear',
                                             max_iter=2000))

clf.fit(X_train_final, y_train)

(6316, 44873)
(1579, 44873)
(6316, 44874)
(1579, 44874)


OneVsRestClassifier(estimator=LogisticRegression(C=3, max_iter=2000,
                                                 solver='liblinear'))

In [20]:
# Predicción del modelo de clasificación

y_pred = clf.predict_proba(X_test_final)

score = roc_auc_score(y_test, y_pred, average='macro')

print('MCAUC_final:', score)

MCAUC VALIDACIÓN: 0.8982801244516376


In [13]:
# Transformación variables predictoras X del conjunto de test
X_test_real_text = dataTesting['text']

X_test_real_word = vect.transform(X_test_real_text)

# variables numéricas
X_test_real_num = dataTesting[['year']].fillna(0)

X_test_real_num_scaled = scaler.transform(X_test_real_num)

X_test_real_num_sparse = csr_matrix(X_test_real_num_scaled)

# combinar
X_test_real_final = hstack([X_test_real_word, X_test_real_num_sparse])

y_pred_test = clf.predict_proba(X_test_real_final)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [14]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(y_pred_test, index=dataTesting.index, columns=cols)

submission.to_csv('RL_TFIDF.csv', index_label='ID')

files.download("RL_TFIDF.csv")
print("RL_TFIDF.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RL_TFIDF.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.099672     0.056597     0.020934     0.027658  0.268511  0.088396   
4  0.063211     0.029916     0.018509     0.159777  0.199618  0.269694   
5  0.081660     0.013481     0.007020     0.057466  0.079910  0.743106   
6  0.041104     0.058727     0.005460     0.038543  0.118772  0.039322   
7  0.027813     0.040887     0.015698     0.017440  0.158293  0.103793   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.024984  0.578520  0.031985   0.113817  ...   0.019516   0.063491   
4       0.015064  0.862582  0.029785   0.019058  ...   0.024506   0.029892   
5       0.016826  0.840068  0.008489   0.019649  ...   0.009980   0.465295   
6       0.003424  0.798027  0.026967   0.034905  ...   0.048299   0.074185   
7       0.003755  0.326076  0.045614   0.103917  ...   0.033138   0.090091   

     p_News  p_Romance  p_Sci-Fi   p_Short   p_Spor

## Regresión Logística Transformer

In [6]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [7]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [8]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train_text, X_val_text, y_train, y_val = train_test_split(dataTraining['text'],
                                                            y,
                                                            test_size=0.2,
                                                            random_state=42)

In [9]:
# Definición y entrenamiento

model_embedding = SentenceTransformer('all-MiniLM-L6-v2')

## Embeddings
X_train_embed = model_embedding.encode(X_train_text.tolist(),
                                       show_progress_bar=True,
                                       batch_size=32)

X_val_embed = model_embedding.encode(X_val_text.tolist(),
                                     show_progress_bar=True,
                                     batch_size=32)

## Modelo
clf = OneVsRestClassifier(LogisticRegression(C=3, max_iter=3000))

clf.fit(X_train_embed, y_train)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/198 [00:00<?, ?it/s]

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

OneVsRestClassifier(estimator=LogisticRegression(C=3, max_iter=3000))

In [10]:
# Predicción del modelo de clasificación

y_pred_val = clf.predict_proba(X_val_embed)

score = roc_auc_score(y_val, y_pred_val, average='macro')

print("MCAUC_t:", score)

MCAUC_t: 0.8926982599881584


In [11]:
# Transformación variables predictoras X del conjunto de test

X_test_embed = model_embedding.encode(dataTesting['text'].tolist(),
                                      show_progress_bar=True,
                                      batch_size=32)

y_pred_test = clf.predict_proba(X_test_embed)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

Batches:   0%|          | 0/106 [00:00<?, ?it/s]

In [12]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(y_pred_test, index=dataTesting.index, columns=cols)

submission.to_csv('RL_Transformer.csv', index_label='ID')

files.download("RL_Transformer.csv")
print("RL_Transformer.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RL_Transformer.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.025791     0.025247     0.001672     0.017246  0.224751  0.106244   
4  0.077130     0.020057     0.002918     0.146476  0.128451  0.410431   
5  0.184895     0.018031     0.001057     0.114771  0.039994  0.731090   
6  0.096165     0.063712     0.002144     0.048001  0.104117  0.048843   
7  0.020337     0.010888     0.011025     0.008217  0.138887  0.086040   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.002792  0.851304  0.001422   0.038634  ...   0.005219   0.066037   
4       0.003316  0.725864  0.006716   0.063310  ...   0.014123   0.027041   
5       0.011144  0.672158  0.001721   0.021693  ...   0.000250   0.479764   
6       0.003502  0.717523  0.003311   0.011465  ...   0.002831   0.077351   
7       0.003058  0.154699  0.018535   0.095268  ...   0.003310   0.639341   

     p_News  p_Romance  p_Sci-Fi   p_Short   

## Regresión Logística

In [6]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [7]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [10]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train, X_val, y_train, y_val = train_test_split(dataTraining['text'],
                                                            y,
                                                            test_size=0.2,
                                                            random_state=42)

In [11]:
# Definición y entrenamiento

## TF-IDF
tfidf = TfidfVectorizer(stop_words='english',
                        lowercase=True,
                        max_features=60000,
                        ngram_range=(1,3),
                        min_df=2,
                        max_df=0.95,
                        sublinear_tf=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_val_tfidf = tfidf.transform(X_val)

print(X_train_tfidf.shape)
print(X_val_tfidf.shape)

## Modelo
clf = OneVsRestClassifier(LogisticRegression(C=5,
                       solver='liblinear',
                       max_iter=4000))

clf.fit(X_train_tfidf, y_train)



(6316, 44873)
(1579, 44873)


OneVsRestClassifier(estimator=LogisticRegression(C=5, max_iter=4000,
                                                 solver='liblinear'))

In [12]:
# Predicción del modelo de clasificación

y_pred_val = clf.predict_proba(X_val_tfidf)

auc_RL2 = roc_auc_score(y_val,y_pred_val,average='macro')

print('MCAUC:', auc_RL2)

MCAUC: 0.8968899266633231


In [13]:
# Transformación variables predictoras X del conjunto de test

X_test_tfidf = tfidf.transform(dataTesting['text'])

y_pred_test = clf.predict_proba(X_test_tfidf)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [14]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(y_pred_test, index=dataTesting.index, columns=cols)

submission.to_csv('RL_TFIDF.csv', index_label='ID')

files.download("RL_TFIDF.csv")
print("RL_TFIDF.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RL_TFIDF.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.073993     0.042343     0.013993     0.020110  0.220676  0.072067   
4  0.071363     0.020410     0.018833     0.174874  0.218042  0.278380   
5  0.055255     0.007504     0.003927     0.051067  0.049586  0.804556   
6  0.049824     0.045193     0.005776     0.039118  0.102707  0.024401   
7  0.024383     0.027173     0.017515     0.013721  0.141861  0.084758   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.017333  0.604890  0.022221   0.102865  ...   0.027940   0.049913   
4       0.038760  0.880898  0.022461   0.013721  ...   0.014687   0.022640   
5       0.012635  0.873090  0.004318   0.011710  ...   0.012211   0.502887   
6       0.024719  0.819522  0.019808   0.031814  ...   0.021046   0.065312   
7       0.013054  0.282419  0.035864   0.101312  ...   0.016718   0.074179   

     p_News  p_Romance  p_Sci-Fi   p_Short   p_Spor

## Regresión logística

In [6]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [7]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [8]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train_text, X_val_text, y_train, y_val = train_test_split(dataTraining['text'],
                                                            y,
                                                            test_size=0.2,
                                                            random_state=42)

In [9]:
# Definición y entrenamiento

## TF-IDF
tfidf = TfidfVectorizer(stop_words='english',
                        lowercase=True,
                        max_features=60000,
                        ngram_range=(1,3),
                        min_df=2,
                        sublinear_tf=True)

X_train = tfidf.fit_transform(X_train_text)
X_val = tfidf.transform(X_val_text)

print(X_train.shape)
print(X_val.shape)

## RL
clf = OneVsRestClassifier(LogisticRegression(C=3,
                                             solver='liblinear',
                                             max_iter=4000))

clf.fit(X_train, y_train)

(6316, 44873)
(1579, 44873)


OneVsRestClassifier(estimator=LogisticRegression(C=3, max_iter=4000,
                                                 solver='liblinear'))

In [10]:
# Predicción del modelo de clasificación

y_pred_val = clf.predict_proba(X_val)

score = roc_auc_score(y_val, y_pred_val, average='macro')

print('MCAUC_RL_TFIDF:', score)

MCAUC_RL_TFIDF: 0.898020220543153


In [11]:
# Transformación variables predictoras X del conjunto de test

X_test = tfidf.transform(dataTesting['text'])

y_pred_test = clf.predict_proba(X_test)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [12]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(y_pred_test, index=dataTesting.index, columns=cols)

submission.to_csv('RL_TFIDF.csv', index_label='ID')

files.download("RL_TFIDF.csv")
print("RL_TFIDF.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RL_TFIDF.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.086369     0.055907     0.018916     0.026585  0.255192  0.088132   
4  0.088526     0.030457     0.023586     0.170689  0.226057  0.271026   
5  0.071323     0.013331     0.006541     0.055476  0.073337  0.742477   
6  0.068206     0.060586     0.009318     0.044040  0.136432  0.039633   
7  0.038988     0.041718     0.023269     0.019007  0.170170  0.104201   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.023226  0.592805  0.031528   0.109406  ...   0.032677   0.062156   
4       0.047821  0.847381  0.030462   0.020801  ...   0.019611   0.031186   
5       0.017710  0.848620  0.008372   0.018723  ...   0.016144   0.459094   
6       0.030889  0.775681  0.028087   0.040484  ...   0.025966   0.078528   
7       0.018864  0.312316  0.046904   0.111794  ...   0.021471   0.093231   

     p_News  p_Romance  p_Sci-Fi   p_Short   p_Spor

## Regresión Logística word + char

In [6]:
# Definición de variables predictoras (X)

dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [7]:
# Definición de variable de interés (y)

dataTraining['genres'] = dataTraining['genres'].map(eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [8]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split

X_train_text, X_val_text, y_train, y_val, idx_train, idx_val = train_test_split(dataTraining['text'],
                                                                                y,
                                                                                dataTraining.index,
                                                                                test_size=0.2,
                                                                                random_state=42)

In [9]:
# Definición y entrenamiento

## TF-IDF Word
tfidf_word = TfidfVectorizer(stop_words='english',
                             lowercase=True,
                             max_features=40000,
                             ngram_range=(1,3),
                             min_df=2,
                             sublinear_tf=True)

X_train_word = tfidf_word.fit_transform(X_train_text)

X_val_word = tfidf_word.transform(X_val_text)

## TF-IDF Char
tfidf_char = TfidfVectorizer(analyzer='char_wb',
                             ngram_range=(3,6),
                             max_features=20000,
                             sublinear_tf=True)

X_train_char = tfidf_char.fit_transform(X_train_text)

X_val_char = tfidf_char.transform(X_val_text)

## Variables numéricas escaladas
scaler_year = StandardScaler() #Año

X_train_year = scaler_year.fit_transform(dataTraining.loc[idx_train, ['year']])

X_val_year = scaler_year.transform(dataTraining.loc[idx_val, ['year']])

## Combinación
X_train_final = hstack([X_train_word,
                        X_train_char,
                        X_train_year])

X_val_final = hstack([X_val_word,
                      X_val_char,
                      X_val_year])

## Modelo Regresión Logística
clf = OneVsRestClassifier(LogisticRegression(C=3,
                       solver='saga',
                       max_iter=5000,
                       n_jobs=-1))

clf.fit(X_train_final, y_train)

KeyboardInterrupt: 

In [ ]:
# Predicción del modelo de clasificación

y_pred_val = clf.predict_proba(X_val_final)

auc_RL = roc_auc_score(y_val, y_pred_val, average='macro')

print("MCAUC RL:", auc_RL)

In [ ]:
# Transformación variables predictoras X del conjunto de test

X_test_word = tfidf_word.transform(dataTesting['text'])

X_test_char = tfidf_char.transform(dataTesting['text'])

X_test_year = scaler_year.transform(dataTesting[['year']])

X_test_final = hstack([X_test_word,
                       X_test_char,
                       X_test_year])

y_pred_test = clf.predict_proba(X_test_final)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [ ]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(y_pred_test, index=dataTesting.index, columns=cols)

submission.to_csv('RL(OneVsRest).csv', index_label='ID')

files.download("RL(OneVsRest).csv")
print("RL(OneVsRest).csv generado y descargado")
print(submission.head())

## Ensemble TF-IDF + RL (OneVsRest)+ NB

In [ ]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [ ]:
# Definición de variable de interés (y)
mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [ ]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(dataTraining['text'],
                                                              y,
                                                              test_size=0.2,
                                                              random_state=42)



In [ ]:
# Definición y entrenamiento

## TF-IDF
vect = TfidfVectorizer(stop_words='english',
                       max_features=15000,
                       ngram_range=(1,2))

X_train_dtm = vect.fit_transform(X_train_text)

X_test_dtm = vect.transform(X_test_text)

## Regresión Logística
lr = OneVsRestClassifier(LogisticRegression(C=2,max_iter=3000))

lr.fit(X_train_dtm, y_train)

pred_lr = lr.predict_proba(X_test_dtm)

## Naive Bayes Multinomial
nb = OneVsRestClassifier(MultinomialNB(alpha=0.1))

nb.fit(X_train_dtm, y_train)

pred_nb = nb.predict_proba(X_test_dtm)

In [ ]:
# Predicción del modelo de clasificación

## Ensemble
pred_ensemble = (0.8 * pred_lr + 0.2 * pred_nb)

auc_TFIDF_RL_NB = roc_auc_score(y_test, pred_ensemble, average='macro')

print('ROC AUC Ensemble TF-IDF + RL (OneVsRest)+ NB:', auc_TFIDF_RL_NB)

ROC AUC Ensemble TF-IDF + RL (OneVsRest)+ NB: 0.8890777461549292


In [ ]:
# Transformación variables predictoras X del conjunto de test
X_real = vect.transform(dataTesting['text'])

pred_lr_test = lr.predict_proba(X_real)

pred_nb_test = nb.predict_proba(X_real)

pred_final = (0.8 * pred_lr_test + 0.2 * pred_nb_test)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [ ]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(pred_final, index=dataTesting.index, columns=cols)

submission.to_csv('Ensemble TFIDF_RL (OneVsRest)_NB.csv', index_label='ID')

files.download("Ensemble TFIDF_RL (OneVsRest)_NB.csv")
print("Ensemble TFIDF_RL (OneVsRest)_NB.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Ensemble TFIDF_RL (OneVsRest)_NB.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.097983     0.063335     0.019165     0.026167  0.313276  0.092109   
4  0.117627     0.033690     0.022907     0.149439  0.270935  0.263628   
5  0.081277     0.018597     0.007469     0.051897  0.127465  0.658406   
6  0.062611     0.049712     0.008795     0.037164  0.157816  0.041908   
7  0.032787     0.035722     0.022129     0.016270  0.198986  0.096925   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.023769  0.554706  0.038718   0.111854  ...   0.033732   0.070314   
4       0.038908  0.784446  0.028160   0.023936  ...   0.019508   0.031342   
5       0.020415  0.804303  0.010966   0.017275  ...   0.013715   0.407397   
6       0.031444  0.790828  0.026861   0.031664  ...   0.025804   0.071058   
7       0.017159  0.232497  0.045190   0.101290  ...   0.020575   0.144565   

     p_News  p_Romance  p_S

## Logistic Regression OneVsRest

In [ ]:
# Definición de variables predictoras (X)
dataTraining['text'] = (dataTraining['title'].fillna('') + ' ' + dataTraining['plot'].fillna(''))

dataTesting['text'] = (dataTesting['title'].fillna('') + ' ' + dataTesting['plot'].fillna(''))

In [ ]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(eval)

mlb = MultiLabelBinarizer()

y = mlb.fit_transform(dataTraining['genres'])

In [ ]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train_text, X_test_text, y_train, y_test = train_test_split(dataTraining['text'],
                                                              y,
                                                              test_size=0.2,
                                                              random_state=42)

In [ ]:
# Definición y entrenamiento

## Capturar palabras y frases
word_vectorizer = TfidfVectorizer(stop_words='english',
                                  max_features=15000,
                                  ngram_range=(1,2),
                                  min_df=3,
                                  max_df=0.9,
                                  sublinear_tf=True)

X_train_word = word_vectorizer.fit_transform(X_train_text)

X_test_word = word_vectorizer.transform(X_test_text)

## Capturar fragmentos de palabras
char_vectorizer = TfidfVectorizer(analyzer='char',
                                  ngram_range=(3,5),
                                  max_features=10000,
                                  sublinear_tf=True)

X_train_char = char_vectorizer.fit_transform(X_train_text)

X_test_char = char_vectorizer.transform(X_test_text)

## Extraer train/test numérico
X_train_num = dataTraining.loc[X_train_text.index, ['year']]
X_test_num = dataTraining.loc[X_test_text.index, ['year']]

## Reemplazar nulos
X_train_num = X_train_num.fillna(0)

X_test_num = X_test_num.fillna(0)

## Escalar
scaler = StandardScaler()

X_train_num_scaled = scaler.fit_transform(X_train_num)

X_test_num_scaled = scaler.transform(X_test_num)

## Combinación de features
X_train_final = hstack([X_train_word,
                        X_train_char,
                        X_train_num_scaled])

X_test_final = hstack([X_test_word,
                       X_test_char,
                       X_test_num_scaled])

# Modelo
clf = OneVsRestClassifier(LogisticRegression(C=4,
                                             solver='liblinear',
                                             max_iter=3000,
                                             class_weight='balanced'))

clf.fit(X_train_final, y_train)

OneVsRestClassifier(estimator=LogisticRegression(C=4, class_weight='balanced',
                                                 max_iter=3000,
                                                 solver='liblinear'))

In [ ]:
# Predicción del modelo de clasificación
y_pred = clf.predict_proba(X_test_final)

## Evaluación AUC
auc_RL_OVS = roc_auc_score(y_test, y_pred, average='macro')

print("ROC AUC Logistic Regression OneVsRest:", auc_RL_OVS)

ROC AUC Logistic Regression OneVsRest: 0.8989262970553126


In [ ]:
# Transformación variables predictoras X del conjunto de test
X_real_text = dataTesting['text']  #Texto
X_real_word = word_vectorizer.transform(X_real_text) #WORD TF-IDF
X_real_char = char_vectorizer.transform(X_real_text) #CHAR TF-IDF

## Variables numéricas
X_real_num = dataTesting[['year']]

X_real_num = X_real_num.fillna(0)

X_real_num_scaled = scaler.transform(X_real_num)

## Combinar test
X_real_final = hstack([X_real_word,
                       X_real_char,
                       X_real_num_scaled])

y_pred_test = clf.predict_proba(X_real_final)

cols = ['p_Action', 'p_Adventure', 'p_Animation',
        'p_Biography', 'p_Comedy', 'p_Crime',
        'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History',
        'p_Horror', 'p_Music', 'p_Musical',
        'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport',
        'p_Thriller', 'p_War', 'p_Western']

In [ ]:
# Guardar predicciones en formato exigido en la competencia de kaggle
submission = pd.DataFrame(y_pred_test,
                          index=dataTesting.index,
                          columns=cols)

submission.to_csv('LRegression_OneVsRest.csv', index_label='ID')

files.download("LRegression_OneVsRest.csv")
print("LR_OneVsRest.csv generado y descargado")
print(submission.head())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

LR_OneVsRest.csv generado y descargado
   p_Action  p_Adventure  p_Animation  p_Biography  p_Comedy   p_Crime  \
1  0.228442     0.082054     0.035289     0.039729  0.203531  0.065832   
4  0.098580     0.007558     0.015565     0.529085  0.122503  0.484638   
5  0.083001     0.008333     0.002479     0.187268  0.026360  0.960070   
6  0.104641     0.156610     0.002188     0.085229  0.108844  0.035551   
7  0.014978     0.066463     0.043217     0.016770  0.236090  0.020454   

   p_Documentary   p_Drama  p_Family  p_Fantasy  ...  p_Musical  p_Mystery  \
1       0.010717  0.308710  0.032106   0.173383  ...   0.053695   0.048969   
4       0.023067  0.969156  0.004645   0.004393  ...   0.034962   0.014933   
5       0.012815  0.733143  0.004150   0.017770  ...   0.006817   0.866140   
6       0.001940  0.831187  0.015596   0.024325  ...   0.053866   0.073151   
7       0.001954  0.385149  0.049744   0.167612  ...   0.028355   0.036544   

     p_News  p_Romance  p_Sci-Fi   p_Short   p_

## Random Forest

In [ ]:
# Definición de variables predictoras (X)
vect = CountVectorizer(max_features=1000)
X_dtm = vect.fit_transform(dataTraining['plot'])
X_dtm.shape

(7895, 1000)

In [ ]:
# Definición de variable de interés (y)
dataTraining['genres'] = dataTraining['genres'].map(lambda x: eval(x))
le = MultiLabelBinarizer()
y_genres = le.fit_transform(dataTraining['genres'])

In [ ]:
# Separación de variables predictoras (X) y variable de interés (y) en set de entrenamiento y test usandola función train_test_split
X_train, X_test, y_train_genres, y_test_genres = train_test_split(X_dtm, y_genres, test_size=0.33, random_state=42)

In [ ]:
# Definición y entrenamiento
clf = OneVsRestClassifier(RandomForestClassifier(n_jobs=-1, n_estimators=100, max_depth=10, random_state=42))
clf.fit(X_train, y_train_genres)

In [ ]:
# Predicción del modelo de clasificación
y_pred_genres = clf.predict_proba(X_test)

# Impresión del desempeño del modelo
roc_auc_score(y_test_genres, y_pred_genres, average='macro')

In [ ]:
# Transformación variables predictoras X del conjunto de test
X_test_dtm = vect.transform(dataTesting['plot'])

cols = ['p_Action', 'p_Adventure', 'p_Animation', 'p_Biography', 'p_Comedy', 'p_Crime', 'p_Documentary', 'p_Drama', 'p_Family',
        'p_Fantasy', 'p_Film-Noir', 'p_History', 'p_Horror', 'p_Music', 'p_Musical', 'p_Mystery', 'p_News', 'p_Romance',
        'p_Sci-Fi', 'p_Short', 'p_Sport', 'p_Thriller', 'p_War', 'p_Western']

# Predicción del conjunto de test
y_pred_test_genres = clf.predict_proba(X_test_dtm)

In [ ]:
# Guardar predicciones en formato exigido en la competencia de kaggle
res = pd.DataFrame(y_pred_test_genres, index=dataTesting.index, columns=cols)
res.to_csv('pred_genres_text_RF.csv', index_label='ID')
files.download("pred_genres_text_RF.csv")
print("pred_genres_text_RF.csv generado y descargado")
print(res.head())